In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0' # Use only the first GPU
os.environ['NCCL_DEBUG']      = 'INFO'
os.environ['NCCL_P2P_DISABLE'] = '1'
os.environ['NCCL_SHM_DISABLE'] = '1'

import torch
import torch.distributed as dist



In [ ]:
import numpy as np
import torchcodec
from datasets import load_dataset, Audio

# Preprocess

In [ ]:
ravdess = load_dataset('amnesiackid/ravdess-emotion-intensity', name='default', split="train")

In [ ]:
def decode_to_array(example):
    decoder = example["audio"]
    samples = decoder.get_all_samples()
    # Average the stereo channels to create a mono (1D) array
    waveform_mono = samples.data.numpy().mean(axis=0)
    return {
        "waveform": waveform_mono,          # NumPy array
        "sampling_rate": samples.sample_rate       # int
    }

ravdess = ravdess.map(
    decode_to_array,
    remove_columns=["audio"],                      # drop the original AudioDecoder
)

In [ ]:
label2id = {"neutral": 0,
    "calm": 1,
    "happy": 2,
    "sad": 3,
    "angry": 4,
    "fearful": 5,
    "disgust": 6,
    "surprised": 7}
id2label = {v: k for k, v in label2id.items()}
def convert_labels(examples):
    # Convert string labels to numeric IDs
    n_emotion = [label2id[label] for label in examples["emotion_labels"]]
    return {"emotion_labels": n_emotion}
# First convert labels to numeric IDs
ravdess = ravdess.map(convert_labels, batched=True)

In [ ]:
# ravdess = ravdess.train_test_split(test_size=0.2)

inspect data structure

In [ ]:

ravdess[0]

In [ ]:
from transformers import Wav2Vec2Model, Wav2Vec2Processor
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
wav2vec2 = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(device)
wav2vec2.eval()  # eval mode

In [ ]:
import os
os.makedirs("checkpoints", exist_ok=True)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

In [ ]:
for i, item in enumerate(ravdess):
    waveform = torch.tensor(item['waveform'], dtype=torch.float32)
    inputs = processor(waveform, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        feat = wav2vec2(inputs.input_values.to(device)).last_hidden_state.squeeze(0).cpu()
    torch.save(feat, f"features/{i}.pt")


# Model training

In [ ]:
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import Wav2Vec2Model, Wav2Vec2Processor
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import LambdaLR




# 1. load wav2vec2
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
wav2vec2 = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(device)
wav2vec2.eval()  

# 2. custom dataset
class EmotionDataset(Dataset):
    def __init__(self, data, processor, wav2vec2):
        self.data = data
        self.processor = processor
        self.wav2vec2 = wav2vec2

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        waveform = torch.tensor(item['waveform'], dtype=torch.float32)
        inputs = self.processor(waveform, sampling_rate=16000, return_tensors="pt", padding=True)
        with torch.no_grad():
            feats = self.wav2vec2(inputs.input_values.to(device)).last_hidden_state.squeeze(0)
        label = torch.tensor(item['emotion_labels'], dtype=torch.long)
        return feats.cpu(), label

def collate_fn(batch):
    feats = [x[0] for x in batch]
    labels = torch.stack([x[1] for x in batch])
    feats_padded = pad_sequence(feats, batch_first=True)  # (B, T_max, D)
    return feats_padded, labels

# 3. split data
dataset = EmotionDataset(ravdess, processor, wav2vec2)
n_total = len(dataset)
n_val = int(n_total * 0.2)      
n_train = n_total - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, collate_fn=collate_fn)
steps_per_epoch = len(train_loader)
# 4. define CNN classifier
class CNNClassifier(nn.Module):
    def __init__(self, feat_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(feat_dim, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.1),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        x = x.transpose(1, 2)  # (B, T, D) → (B, D, T)
        return self.net(x)

n_classes = len({item['emotion_labels'] for item in ravdess})
model = CNNClassifier(feat_dim=wav2vec2.config.hidden_size, n_classes=n_classes).to(device)
optimizer = optim.Adam(model.parameters(), lr=5e-4)
criterion = nn.CrossEntropyLoss()
global_step = 0
epochs = 32
total_steps = epochs * steps_per_epoch
warmup_steps = int(0.05 * total_steps)

def lr_lambda(current_step):
    if current_step < warmup_steps:
        # linear warmup
        return current_step / float(max(1, warmup_steps))
    # cosine decay
    progress = (current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda)

# 5. train and valudate

for epoch in range(epochs):
    # ——— train ———
    model.train()
    train_loss = 0.0
    train_correct = 0
    for feats, labels in train_loader:
        feats, labels = feats.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(feats)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        global_step += 1
        train_loss += loss.item() * feats.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
    train_loss /= n_train
    train_acc = train_correct / n_train

    # ——— validate ———
    model.eval()
    val_loss = 0.0
    val_correct = 0
    with torch.no_grad():
        for feats, labels in val_loader:
            feats, labels = feats.to(device), labels.to(device)
            logits = model(feats)
            loss = criterion(logits, labels)
            val_loss += loss.item() * feats.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
    val_loss /= n_val
    val_acc = val_correct / n_val

    print(
        f"Epoch {epoch}/{epochs} — "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
    )


In [ ]:
# 6. save model
os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/cnn_classifier.pth")   